# AI consumption analysis

## Overview

In [240]:
# to read datafile
import pandas as pd

# to use SQL queries
import duckdb as db

# to make simple charts
import matplotlib.pyplot as plt

In [ ]:
# reads datafile in from another folder
households_electricity_consumption = pd.read_csv('../../DataSources/fictional_households_electricity_consumption_sorted.csv', sep = ";")
households_electricity_consumption

In [ ]:
# overview of datafile values
db.sql("SUMMARIZE SELECT * FROM households_electricity_consumption")

In [ ]:
# counts & names unique id-s
print(households_electricity_consumption['household_id'].unique())

In [244]:
# changes timestamp from str to datetime
households_electricity_consumption['timestamp'] = pd.to_datetime(households_electricity_consumption['timestamp'], format='%d.%m.%Y %H:%M:%S')

# changes date from str to datetime
households_electricity_consumption['date'] = pd.to_datetime(households_electricity_consumption['date'], format='%d.%m.%Y')

# changes time from str to datetime
households_electricity_consumption['time'] = pd.to_datetime(households_electricity_consumption['time'], format='%H:%M:%S')

# changes consumption from str to float with decimal separator normalization
households_electricity_consumption['consumption_kwh'] = households_electricity_consumption['consumption_kwh'].str.replace(',', '.').astype(float)


In [ ]:
# checks column datatypes
households_electricity_consumption.dtypes

## Data aggregation

In [ ]:
# aggregates consumption by year
consumption_per_year = db.query("""
    SELECT
        CAST(YEAR(date) AS INTEGER) AS year,
        SUM(consumption_kwh) AS total_consumption_kwh
    FROM households_electricity_consumption
    GROUP BY YEAR(date)
    ORDER BY year
""").df()

print(consumption_per_year)

In [ ]:
# aggregates consumption by month
consumption_per_month = db.query("""
    SELECT
        CAST(month(date) AS INTEGER) AS month,
        SUM(consumption_kwh) AS total_consumption_kwh
    FROM households_electricity_consumption
    GROUP BY month(date)
    ORDER BY month
""").df()

print(consumption_per_month)

In [ ]:
# aggregates consumption by day
consumption_per_day = db.query("""
    SELECT
        CAST(day(date) AS INTEGER) AS day,
        SUM(consumption_kwh) AS total_consumption_kwh
    FROM households_electricity_consumption
    GROUP BY day(date)
    ORDER BY day
""").df()

print(consumption_per_day)

In [ ]:
# aggregates consumption by hour
consumption_per_hour = db.query("""
    SELECT
        CAST(hour(time) AS INTEGER) AS hour,
        SUM(consumption_kwh) AS total_consumption_kwh
    FROM households_electricity_consumption
    GROUP BY hour(time)
    ORDER BY hour
""").df()

print(consumption_per_hour)

In [ ]:
# aggregates consumption by household_id
consumption_per_household = db.query("""
    SELECT
        household_id AS household,
        SUM(consumption_kwh) AS total_consumption_kwh
    FROM households_electricity_consumption
    GROUP BY household
    ORDER BY household ASC
""").df()

print(consumption_per_household)

## Simple graphs

In [ ]:
# consumption by year graph, converting year into a discrete category / matplot
ax = consumption_per_year.assign(year=consumption_per_year['year'].astype(str)).plot(
    x="year", y="total_consumption_kwh", kind="line")

# defines the y axis starting point as 0
ax.set_ylim(0, None)

# adds a title to the graph
ax.set_title("Total consumption by year", fontsize=13)

# adds a title to y axis
ax.set_ylabel("Total consumption (kWh)")

In [ ]:
# consumption by month graph / matplot
ax = consumption_per_month.plot(
    x="month", y="total_consumption_kwh", kind="line")

# defines the amount of x axis points
ax.set_xticks(range(12))

# defines the labels on x axis points
ax.set_xticklabels(range(1, 13))

# defines the y axis starting point as 0
ax.set_ylim(0, None)

# defines seasons / st-ni
spring_start = 2
spring_end = 5
summer_start = 5
summer_end = 8
autumn_start = 8
autumn_end = 11
winter_start = 11
winter_end = 2

# adds a light color for spring
ax.axvspan(spring_start, spring_end, alpha=0.2, color='green', label='Spring')

# adds a light color for summer
ax.axvspan(summer_start, summer_end, alpha=0.2, color='yellow', label='Summer')

# adds a light color for autumn
ax.axvspan(autumn_start, autumn_end, alpha=0.2, color='brown', label='Autumn')

# adds a light color for winter
ax.axvspan(winter_start, 12, alpha=0.2, color='blue', label = 'Winter')
ax.axvspan(0, winter_end, alpha=0.2, color='blue')

# adds a legend
ax.legend()

# adds a title to the graph
ax.set_title("Total consumption by month", fontsize=13)

# adds a title to y axis
ax.set_ylabel("Total consumption (kWh)")


In [ ]:
# consumption by day graph / matplot
ax = consumption_per_day.plot(
    x="day", y="total_consumption_kwh", kind="line", figsize=[12,5])

# defines the amount of x axis points
ax.set_xticks(range(31))

# defines the labels on x axis points
ax.set_xticklabels(range(1, 32))

# defines the y axis starting point as 0
ax.set_ylim(0, None)

# adds a title to the graph
ax.set_title("Total consumption by day", fontsize=13)

# adds a title to y axis
ax.set_ylabel("Total consumption (kWh)")

In [ ]:
# consumption by hour graph / matplot
ax = consumption_per_hour.plot(
    x="hour", y="total_consumption_kwh", kind="line", figsize=[10,5])

# defines the amount of x axis points
ax.set_xticks(range(24))

# defines the labels on x axis points
ax.set_xticklabels(range(0, 24))

# defines the y axis starting point as 0
ax.set_ylim(0, None)

# defines daytime and nighttime
day_start = 7
night_start = 22

# adds a light color for daytime
ax.axvspan(day_start, night_start, alpha=0.05, color='yellow', label='Daytime')

# adds a light color for nighttime
ax.axvspan(0, day_start, alpha=0.05, color='blue', label='Nighttime')
ax.axvspan(night_start, 23, alpha=0.05, color='blue')

# defines peak hours
morning_peak_start = 7
morning_peak_end = 9
evening_peak_start = 17
evening_peak_end = 20

# adds a light color for peak times in the morning and evening
ax.axvspan(morning_peak_start, morning_peak_end, alpha=0.2, color='red', label='Peak')
ax.axvspan(evening_peak_start, evening_peak_end, alpha=0.2, color='red')

# adds a legend
ax.legend()

# adds a title to the graph
ax.set_title("Total consumption by hour", fontsize=13)

# adds a title to y axis
ax.set_ylabel("Total consumption (kWh)")

In [ ]:
# consumption by household bar chart / matplot
ax = consumption_per_household.plot(
    x="household", y="total_consumption_kwh", kind="bar", figsize=(20, 5))

# defines the y axis starting point as 0
ax.set_ylim(0, None)

# counts the x axis points
len(ax.get_xticks())

# adds a title to the graph
ax.set_title("Total consumption by household", fontsize=13)

# adds a title to y axis
ax.set_ylabel("Total consumption (kWh)")